In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as tt

import graphviz

[Leukemia data](https://vincentarelbundock.github.io/Rdatasets/datasets.html). [Description](https://vincentarelbundock.github.io/Rdatasets/doc/Stat2Data/Leukemia.html)

In [ ]:
leuc = pd.read_csv('Leukemia.csv')

In [ ]:
leuc.head()

## Hierarchical model: age

In [ ]:
ages = leuc.Age.sort_values().unique()
ages

In [ ]:
ages_dict = pd.DataFrame({
    'Age': ages, 'num': range(ages.shape[0])
})
leuc_aux = leuc.merge(ages_dict, how='left')
leuc_aux.head(10)

In [ ]:
N = leuc.shape[0]

with pm.Model() as partial_pool:
    phi = pm.Gamma("phi", alpha=1, beta=1)
    kappa = pm.Gamma("kappa", alpha=1, beta=1)

    lam = pm.Gamma("lambdas", alpha=phi, beta=kappa, shape=ages.shape[0])
    y = pm.Exponential("y", lam=lam[leuc_aux.num], observed=leuc.Time)

In [ ]:
pm.model_to_graphviz(partial_pool)

In [ ]:
with partial_pool:
    trace = pm.sample(2000, tune=200, chains=2, target_accept=0.80, return_inferencedata=True)

    # check convergence diagnostics
    assert all(az.rhat(trace) < 1.03)

In [ ]:
az.plot_trace(trace, var_names=["phi", "kappa"]);

In [ ]:
ax = az.plot_forest(trace, var_names=["lambdas"])
ax[0].set_yticklabels(ages);

In [ ]:
plt.scatter(leuc.Age, leuc.Time)
plt.xlabel('Age')
plt.ylabel('Time')

## Bayesian linear model

In [ ]:
N = leuc.shape[0]

with pm.Model() as simple_linear:

    intercept = pm.Gamma("intercept", alpha=1, beta=1, shape=1)
    betas = pm.Gamma("betas", alpha=1, beta=1, shape=1)
    regression = pm.Deterministic("regression", intercept + betas*leuc.Age)
    y = pm.Exponential("y", lam=regression, observed=leuc.Time)

In [ ]:
with simple_linear:
    trace_lin = pm.sample(5000, tune=200, chains=2, target_accept=0.90, return_inferencedata=True)

    # check convergence diagnostics
    assert all(az.rhat(trace_lin) < 1.03)

In [ ]:
az.plot_trace(trace_lin, var_names=["intercept","betas","regression"])
plt.show()

In [ ]:
trace_lin.posterior.regression.to_numpy().shape

In [ ]:
plt.scatter(
    np.tile(leuc.Age,10000)+
        np.random.rand(np.tile(trace_lin.observed_data.y.to_numpy(),10000).shape[0]),
    trace_lin.posterior.regression.to_numpy().reshape(-1),
    alpha=0.01, s=1
)
plt.ylim(0,0.3)

In [ ]:
pd.DataFrame({
    'age': np.tile(leuc.Age,10000),
    'regression': trace_lin.posterior.regression.to_numpy().reshape(-1)
}).groupby('age').regression.agg([
    ('upper',lambda x: x.quantile(0.975)),
    ('mean', lambda x: x.mean()),
    ('lower',lambda x: x.quantile(0.025))]).plot()
plt.ylabel('Lambda')

In [ ]:
pd.DataFrame({
    'age': np.tile(leuc.Age,10000),
    'regression': 1/trace_lin.posterior.regression.to_numpy().reshape(-1)
}).groupby('age').regression.agg([
    ('upper',lambda x: x.quantile(0.975)),
    ('mean', lambda x: x.mean()),
    ('lower',lambda x: x.quantile(0.025))]).plot()
plt.ylabel('E(y~exp)')

In [ ]:
trace_lin

### Predictive

In [ ]:
samples = pm.sample_posterior_predictive(trace_lin, samples=10000, model=simple_linear)

In [ ]:
samples['y'].shape

In [ ]:
plt.scatter(
    np.tile(leuc.Age,10000)+
        np.random.rand(np.tile(leuc.Age,10000).shape[0]),
    samples['y'].reshape(-1),
    alpha=0.01, s=1
)

In [ ]:
pd.DataFrame({
    'age': np.tile(leuc.Age,10000),
    'regression': samples['y'].reshape(-1)
}).groupby('age').regression.agg([
    ('upper',lambda x: x.quantile(0.975)),
    ('mean', lambda x: x.mean()),
    ('lower',lambda x: x.quantile(0.025))]).plot()
plt.ylabel('Months')
plt.show()

In [ ]:
samples['y']